Imports

In [1]:
import torch
import torch.nn as nn

Encoder

In [2]:
class CNNEncoder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.conv = nn.Sequential(
            # block 1: [B,1,32,128] -> [B,64,16,64]
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # block 2: -> [B,128,8,32]
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # block 3: -> [B,256,4,32]  (height only pooled)
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d((2, 1)),

            # block 4: -> [B,256,2,32]
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d((2, 1)),

            # block 5: -> [B,256,1,32]
            nn.Conv2d(256, d_model, 3, padding=1), nn.BatchNorm2d(d_model), nn.ReLU(),
            nn.MaxPool2d((2, 1)),
        )

    def forward(self, x):
        x = self.conv(x)              # [B, d_model, 1, 32]
        x = x.squeeze(2)               # [B, d_model, 32]
        x = x.permute(0, 2, 1)         # [B, 32, d_model]
        return x

In [3]:
encoder = CNNEncoder(d_model=256)
dummy = torch.randn(4, 1, 32, 128)   # fake batch of 4 images
out = encoder(dummy)
print(out.shape)   # expect [4, 32, 256]

torch.Size([4, 32, 256])


Same stuff from notebook 3

In [4]:
import json
from torch.utils.data import DataLoader
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import Dataset

with open('../src/vocab.json') as f:
    vocab = json.load(f)
stoi = vocab['stoi']
PAD, BOS, EOS = 0, 1, 2

paths, labels = [], []
with open("../data/synth/train/labels.txt") as f:
    for line in f:
        fname, label = line.strip().split("\t")
        paths.append(f"../data/synth/train/{fname}")
        labels.append(label)

transform = T.Compose([
    T.Resize((32, 128)), T.Grayscale(), T.ToTensor(), T.Normalize([0.5], [0.5]),
])

class OCRDataset(Dataset):
    def __init__(self, paths, labels, transform, stoi):
        self.paths, self.labels, self.transform, self.stoi = paths, labels, transform, stoi
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = self.transform(Image.open(self.paths[idx]).convert('RGB'))
        ids = [self.stoi[c] for c in self.labels[idx].lower() if c in self.stoi]
        return img, ids

def collate_fn(batch):
    imgs, label_lists = zip(*batch)
    imgs = torch.stack(imgs)
    max_len = max(len(l) for l in label_lists) + 1
    tgt_in = torch.full((len(batch), max_len), PAD, dtype=torch.long)
    tgt_out = torch.full((len(batch), max_len), PAD, dtype=torch.long)
    for i, ids in enumerate(label_lists):
        seq = [BOS] + ids
        tgt_in[i, :len(seq)] = torch.tensor(seq)
        out = ids + [EOS]
        tgt_out[i, :len(out)] = torch.tensor(out)
    return imgs, tgt_in, tgt_out

ds = OCRDataset(paths, labels, transform, stoi)
loader = DataLoader(ds, batch_size=8, collate_fn=collate_fn, shuffle=True)

In [5]:
imgs, tgt_in, tgt_out = next(iter(loader))
out = encoder(imgs)
print(out.shape)   # expect [8, 32, 256]

torch.Size([8, 32, 256])


In [6]:
n_params = sum(p.numel() for p in encoder.parameters())
print(f"{n_params:,} params")   # should be a few hundred thousand, well under 1M

1,551,744 params
